In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt


from sirf.Utilities import examples_data_path
import sirf.STIR as spect
import os
import re

In [ ]:
def sorted_alphanumeric(data):
    # Function sorting the data by alphabetical order
    convert = lambda text: int(text) if text.isdigit() else text.lower()
    alphanum_key = lambda key: [convert(c) for c in re.split('([0-9]+)', key)]
    return sorted(data, key=alphanum_key)

In [ ]:
def get_images(path_dir_image):
    # Function to open images in the image list
    image_list = sorted_alphanumeric(os.listdir(path_dir_image))
    true_images = []
    for im in image_list:
        image = {'file name' : path_dir_image + im}
        #image = {'file name' : im}
        
        true_images.append(image)
    
    return true_images

In [ ]:
path_dir_image = '/Data/'  # path to numpy images

In [ ]:
file_names = get_images(path_dir_image)

In [ ]:
images = []
for file in file_names:
    im = np.load(file['file name'])
    images.append(im)

plt.imshow(images[0][64])

In [ ]:
# Acquire with resolution modelling for MEGP collimators
def acquire_data(phantom_data, scale_factor = 0.5):  

    # Scale factor can be used to increase/decrease the number of counts in the image, for example mimicking a 
    # longer or shorter time per projection
    
    acq_model_matrix_sim = spect.SPECTUBMatrix();
   # acq_model_matrix_sim.set_resolution_model(collimator_sigma_0, collimator_slope, full_3D=False)    
    acq_model_sim = spect.AcquisitionModelUsingMatrix(acq_model_matrix_sim) 

    # Create an empty image based on the dimensions of the template sinogram
    # require same number slices and equal z-sampling for projection data & image
    image = templ_sino.create_uniform_image()
    image = image.zoom_image(zooms=(0.5, 1.0, 1.0), size=(templ_sino.dimensions()[1],templ_sino.dimensions()[3], templ_sino.dimensions()[3]))
    #shape = image.as_array().shape

    # Project the image to obtain simulated acquisition data
    acq_model_sim.set_up(templ_sino, image)
    simulated_data = templ_sino.get_uniform_copy()
    noisy_simulated_data = templ_sino.get_uniform_copy()   

    # Fill the template image with 'real' phantom data
    phantom = image.fill(phantom_data)
      
    # Create a noisy representation of the phantom data (Poisson noise) 
    noisy_array=np.random.poisson(phantom.as_array()*scale_factor).astype('float64')

    # Fill the noisy phantom with the noisy data 
    noisy_phantom = phantom.clone()
    noisy_phantom = noisy_phantom.fill(noisy_array);

    # Forward-project and save sinograms for the 'true' and 'noisy' phantoms
    true_sinogram = acq_model_sim.forward(phantom)
    noisy_sinogram = acq_model_sim.forward(noisy_phantom)
    
    return true_sinogram, noisy_sinogram


In [ ]:
def reconstruct_data(sinogram):

    """
    Reconstruct the sinogram
    """
    
    image = templ_sino.create_uniform_image()
    image = image.zoom_image(zooms=(0.5, 1.0, 1.0), size=(templ_sino.dimensions()[1],templ_sino.dimensions()[3], templ_sino.dimensions()[3]))

    # Reconstruct without resolution modelling to prevent 'inverse crime'
    acq_model_matrix_recon = spect.SPECTUBMatrix();
    acq_model_matrix_recon.set_keep_all_views_in_cache(True)
    acq_model_recon = spect.AcquisitionModelUsingMatrix(acq_model_matrix_recon)

    # Reconstruct the sinogram with the reconstruction acquisition model

    obj_fun = spect.make_Poisson_loglikelihood(sinogram)
    obj_fun.set_acquisition_model(acq_model_recon)

    num_subsets = 2 # number of subsets for OSEM reconstruction
    num_subiters = 24 #number of subiterations (i.e two full iterations)

    recon = spect.OSMAPOSLReconstructor()
    recon.set_objective_function(obj_fun)
    recon.set_num_subsets(num_subsets)
    recon.set_num_subiterations(num_subiters)

    # Reconstruct sinogram
    init_image = image.get_uniform_copy(1)

    recon.set_current_estimate(init_image)
    recon.set_up(init_image)
    recon.process()
    reconstructed_image = recon.get_output()
   
    return reconstructed_image

In [ ]:
templ_sino = spect.AcquisitionData('/Data/')  # path to your template sinogram


## Run functions

In [ ]:

test_distribution = images[0]
plt.imshow(test_distribution[64])

true_sinogram, noisy_sinogram = acquire_data(test_distribution,  scale_factor = 0.5)

In [ ]:
true_sinogram.show(64)

In [ ]:
noisy_sinogram.show(64)

In [ ]:
noisy_recon = reconstruct_data(noisy_sinogram)
noisy_recon.show(64)